### Using gemini api for initial implementation of the agent

In [10]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)
MODEL = "gemini-3.5-flash-lite"

In [3]:
# Testing api key
response = client.models.generate_content(
    model=MODEL,
    contents="What is 17 + 25?"
)
print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


17 + 25 = 42


#### Checking Response to analyse the type, data and seeing the empty array of the automatic function calls

In [4]:
print(type(response))
print(response)

<class 'google.genai.types.GenerateContentResponse'>
sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text='17 + 25 = 42',
        thought_signature=b"\x12^\n\\\x01\x11M2\x0f\xecc;\xe1\x95\x857!\xa1/\xb2\xda\x04\xb3\xcc\x122\xbcl8\xda1s{\n'\x83\x9e~<(\xea\xd9\xf2i[\xd8.\x1b\x01+p\xb69f\xden_\xd9|\x83\xee+?-$,\xa0\xd4\xc8\xc7^~\xe6i\x9f\xfa\x85\xdeF\xe5/ka\xa1\xb4B,\x18\xfdr0\xc3\xea\xab/z"
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.5-flash-lite' prompt_feedback=None response_id='NzWsaqa0Da6smNMPn6bD4Ak' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=10,
  prompt_token_count=11,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=11
    ),
  ],
  total_token_count=21
) model_status=None automatic_function_calling_history=[

In [11]:
def add_numbers(a:int, b:int)->int:
    "Adds two numbers together and returns their sum."
    return a+b

In [6]:
response = client.models.generate_content(
    model=MODEL,
    contents="What is 17 + 25?",
    config={
        "tools": [add_numbers]
    }
)

In [7]:
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text='17 + 25 = 42',
        thought_signature=b'\x12^\n\\\x01\x11M2\x0f\t\xa0V\x9c\xb2\x04z|k\xc4\\\x00\x0bG\x06c\xce\xbd\xe6\xde\xbf2\xda\xcbi8\xc2\x16\x0b\x81\x08%\x92\xf8{\x0c\xb1\x80\xef\xa8u\xe2P$\xc8\xb3Wej\x00\xa0\x17\x1aMhc^\xf8\x9f\xcc\x04\x91GOZ\xd4\xc0\xfb\x05\x89\xeeK`[\xdeH\n\xda\xed\x8a\x10P\xa7BI\x91\xa5'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.5-flash-lite' prompt_feedback=None response_id='PDWsasi8Jaexg8UPg46SYQ' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=10,
  prompt_token_count=110,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=110
    ),
  ],
  total_token_count=120
) model_status=None automatic_function_calling_history=[UserContent(
  part

In [8]:
print(response.text)

17 + 25 = 42


### Understood: automatic function calling is by default enabled for gemini, but want to implement the function calling by scratch so that gemini can be easily replaced by other models and to better understand how tools are called. Hence disabled it from here to implement it manually

In [5]:
from google.genai import types
config = types.GenerateContentConfig(
    tools=[add_numbers],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    )
)
response = client.models.generate_content(
    model=MODEL,
    contents="What is 17 + 25?",
    config=config
)

In [9]:
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'a': 17,
            'b': 25
          },
          id='call_1152867',
          name='add_numbers'
        ),
        thought_signature=b"\x12^\n\\\x01\x11M2\x0f\x02\xe6E\xd9nQ\xc6\x80\x0b/\x88\xb08K\xfb\xe9\x02\xbcOu'\x86\xf2\xd0\xc0'\xf1\xfb\x9a\xc1~\x11\xa3\xda\x1b\xbf\x93@;q\x8eP\x1e\x90\xe9\xc6\x94l=\xa0:\xb7\xf4r\xa4\xa0\xe2\xf7`\xbb\xdd\x9e_\x8c\x15\xc5\x10\x81\xa0\xf9G&G\x16\xa9\xd9>\x8cX\xbe\xfb\x9f\x90\xd4Fcq"
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.5-flash-lite' prompt_feedback=None response_id='d-irapSOOoTf4-EPlray2Qo' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=20,
  prompt_token_count=63,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<Med

#### Disabled automatic function calling, and saw how the function call is stored with args, id and name

In [29]:
print(response.text)

17 + 25 = 42


In [30]:
candidate = response.candidates[0]

In [31]:
content = candidate.content

In [32]:
parts = content.parts

In [33]:
part = parts[0]

In [ ]:
print(part)

In [ ]:
print(part.function_call)

In [36]:
function_call = part.function_call

In [37]:
function_call

In [ ]:
if function_call.name == "add_numbers":
    result = add_numbers(**function_call.args)

print(result)

In [ ]:
tool_response = types.Part.from_function_response(
    name=function_call.name,
    response={
        "result": result
    }
)

print(tool_response)

In [26]:
contents = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(
                text="What is 17 + 25?"
            )
        ]
    ),

    response.candidates[0].content,

    types.Content(
        role="user",
        parts=[tool_response]
    )
]

In [27]:
final_response = client.models.generate_content(
    model=MODEL,
    contents=contents,
    config=types.GenerateContentConfig(
        tools=[add_numbers],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=True
        )
    )
)

In [28]:
print(final_response.text)

17 + 25 = 42


In [29]:
print(contents)

[Content(
  parts=[
    Part(
      text='What is 17 + 25?'
    ),
  ],
  role='user'
), Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'a': 17,
          'b': 25
        },
        id='call_1152867',
        name='add_numbers'
      ),
      thought_signature=b"\x12^\n\\\x01\x11M2\x0f\x02\xe6E\xd9nQ\xc6\x80\x0b/\x88\xb08K\xfb\xe9\x02\xbcOu'\x86\xf2\xd0\xc0'\xf1\xfb\x9a\xc1~\x11\xa3\xda\x1b\xbf\x93@;q\x8eP\x1e\x90\xe9\xc6\x94l=\xa0:\xb7\xf4r\xa4\xa0\xe2\xf7`\xbb\xdd\x9e_\x8c\x15\xc5\x10\x81\xa0\xf9G&G\x16\xa9\xd9>\x8cX\xbe\xfb\x9f\x90\xd4Fcq"
    ),
  ],
  role='model'
), Content(
  parts=[
    Part(
      function_response=FunctionResponse(
        name='add_numbers',
        response={
          'result': 42
        }
      )
    ),
  ],
  role='user'
)]


#### Trying multiple tools

In [62]:
contents = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(
                text="What is 17 + 25? and then multiply result by 10"
            )
        ]
    )
]

In [49]:
while True:

    response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            tools=tools,
            automatic_function_calling=types.AutomaticFunctionCallingConfig(
                disable=True
            )
        )
    )

    part = response.candidates[0].content.parts[0]

    # Model gave us a normal response
    if part.text:
        print("FINAL:", part.text)
        break

    # Model requested a tool
    if part.function_call:
        function_call = part.function_call

        print("TOOL CALL:")
        print("  name:", function_call.name)
        print("  args:", function_call.args)

        # Execute the tool
        # if function_call.name == "add_numbers":
        #     result = add_numbers(**function_call.args)

        tool = tool_registry[function_call.name]
        result = tool(**function_call.args)

        print("TOOL RESULT:", result)

        # Add model's tool call to conversation
        contents.append(
            response.candidates[0].content
        )

        # Add tool result to conversation
        contents.append(
            types.Content(
                role="user",
                parts=[
                    types.Part.from_function_response(
                        name=function_call.name,
                        response={
                            "result": result
                        }
                    )
                ]
            )
        )

TOOL CALL:
  name: add_numbers
  args: {'b': 25, 'a': 17}
TOOL RESULT: 42
TOOL CALL:
  name: multiply_numbers
  args: {'b': 10, 'a': 42}
TOOL RESULT: 420
FINAL: 17 + 25 = 42. 
Multiplying that result by 10 gives **420**.


In [12]:
def multiply_numbers(a: int, b: int) -> int:
    "Multiplies two numbers together and returns their product."
    return a * b

In [12]:
tools = [
    add_numbers,
    multiply_numbers
]

In [13]:
# Runtime registry
tool_registry = {
    "add_numbers": add_numbers,
    "multiply_numbers": multiply_numbers,
}

#### Creating a tool class so that I can store name, function and description in efficient way, and also retrieve the parameters and execute
#### Other than that, also made a function to automatically get list of tools to pass to gemini

In [13]:
from dataclasses import dataclass
from typing import Callable, Any, get_type_hints
import inspect

from google.genai import types


@dataclass
class Tool:
    name: str
    description: str
    function: Callable[..., Any]

    def execute(self, **kwargs):
        return self.function(**kwargs)

    def get_parameters(self):
        signature = inspect.signature(self.function)
        type_hints = get_type_hints(self.function)

        properties = {}
        required = []

        for param_name, param in signature.parameters.items():

            if param.kind in (
                inspect.Parameter.VAR_POSITIONAL,
                inspect.Parameter.VAR_KEYWORD,
            ):
                continue

            python_type = type_hints.get(param_name, str)

            if python_type == int:
                json_type = "integer"
            elif python_type == float:
                json_type = "number"
            elif python_type == bool:
                json_type = "boolean"
            else:
                json_type = "string"

            properties[param_name] = {
                "type": json_type
            }

            if param.default is inspect.Parameter.empty:
                required.append(param_name)

        return {
            "type": "object",
            "properties": properties,
            "required": required,
        }

    def to_gemini_declaration(self):
        parameters = self.get_parameters()

        return types.FunctionDeclaration(
            name=self.name,
            description=self.description,
            parameters=types.Schema(
                type="OBJECT",
                properties={
                    name: types.Schema(
                        type=info["type"].upper()
                    )
                    for name, info in parameters["properties"].items()
                },
                required=parameters["required"],
            ),
        )

In [14]:
add_tool = Tool(
    name="add_numbers",
    description="Adds two numbers together and returns their sum.",
    function=add_numbers
)

multiply_tool = Tool(
    name="multiply_numbers",
    description="Multiplies two numbers together and returns the product.",
    function=multiply_numbers
)

In [15]:
tool_registry = {
    tool.name: tool
    for tool in [add_tool, multiply_tool]
}

In [16]:
gemini_tool = types.Tool(
    function_declarations=[
        tool.to_gemini_declaration()
        for tool in tool_registry.values()
    ]
)

In [17]:
config = types.GenerateContentConfig(
    tools=[gemini_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    )
)

In [63]:
while True:

    response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=config,
    )

    model_content = response.candidates[0].content

    contents.append(model_content)

    tool_calls = [
        part.function_call
        for part in model_content.parts
        if part.function_call
    ]

    if not tool_calls:

        for part in model_content.parts:
            if part.text:
                print("FINAL:", part.text)

        break

    tool_results = []

    for function_call in tool_calls:

        print("TOOL CALL:")
        print("  name:", function_call.name)
        print("  args:", function_call.args)

        tool = tool_registry[function_call.name]

        result = tool.execute(**function_call.args)

        print("TOOL RESULT:", result)

        tool_results.append(
            types.Part.from_function_response(
                name=function_call.name,
                response={
                    "result": result
                }
            )
        )

    contents.append(
        types.Content(
            role="user",
            parts=tool_results
        )
    )

TOOL CALL:
  name: add_numbers
  args: {'b': 25, 'a': 17}
TOOL RESULT: 42
TOOL CALL:
  name: multiply_numbers
  args: {'b': 10, 'a': 42}
TOOL RESULT: 420
FINAL: 17 + 25 = 42, and multiplying that result by 10 gives 420.


#### Decided to create ToolRegistry class so that i can easly add, list, execute tools 

In [48]:
class ToolRegistry:

    def __init__(self):
        self.tools = {}

    def register(self, tool: Tool):
        self.tools[tool.name] = tool

    def get(self, name: str) -> Tool:
        return self.tools[name]

    def list_tools(self):
        return list(self.tools.values())

    def execute(self, name: str, **kwargs):
        try:
            tool = self.get(name)
            result = tool.execute(**kwargs)
            return {
                "success": True,
                "result": result
            }

        except Exception as e:
            return {
                "success": False,
                "error": type(e).__name__,
                "message": str(e)
            }

    def to_gemini_tool(self):
        return types.Tool(
            function_declarations=[
                tool.to_gemini_declaration()
                for tool in self.tools.values()
            ]
        )

In [49]:
tool_registry = ToolRegistry()

tool_registry.register(add_tool)
tool_registry.register(multiply_tool)

In [50]:
print(tool_registry.list_tools())

[Tool(name='add_numbers', description='Adds two numbers together and returns their sum.', function=<function add_numbers at 0x000001FA81DE3D80>), Tool(name='multiply_numbers', description='Multiplies two numbers together and returns the product.', function=<function multiply_numbers at 0x000001FA81DE3C40>)]


In [21]:
gemini_tool = tool_registry.to_gemini_tool()

In [16]:
config = types.GenerateContentConfig(
    tools=[tool_registry.to_gemini_tool()],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    )
)

In [41]:
contents = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(
                text="What is 17 + 25, and then multiply the result by 10?"
            )
        ]
    )
]

In [43]:
while True:

    response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=config,
    )

    model_content = response.candidates[0].content
    contents.append(model_content)

    tool_calls = [
        part.function_call
        for part in model_content.parts
        if part.function_call
    ]

    if not tool_calls:

        for part in model_content.parts:
            if part.text:
                print("FINAL:", part.text)

        break

    tool_results = []

    for function_call in tool_calls:

        print("TOOL CALL:")
        print("  name:", function_call.name)
        print("  args:", function_call.args)

        result = tool_registry.execute(
            function_call.name,
            **function_call.args
        )

        print("TOOL RESULT:", result)

        tool_results.append(
            types.Part.from_function_response(
                name=function_call.name,
                response={"result": result}
            )
        )

    contents.append(
        types.Content(
            role="user",
            parts=tool_results
        )
    )

TOOL CALL:
  name: add_numbers
  args: {'a': 17, 'b': 25}
TOOL RESULT: 42
TOOL CALL:
  name: multiply_numbers
  args: {'a': 42, 'b': 10}
TOOL RESULT: 420
FINAL: First, adding 17 and 25 gives 42. 

Then, multiplying that result by 10 gives **420**.


#### For file operations, decided to create a safe workspace to experiment in

In [22]:
from pathlib import Path

WORKSPACE = Path.cwd().parent / "workspace"
def resolve_path(path: str) -> Path:
    target = (WORKSPACE / path).resolve()

    if not target.is_relative_to(WORKSPACE.resolve()):
        raise ValueError("Path is outside the workspace.")

    return target

In [4]:
resolve_path("hello.py")

WindowsPath('D:/ml/nyi/agent-learning/workspace/hello.py')

In [23]:
def list_files(path: str = ".") -> str:
    directory = resolve_path(path)

    if not directory.exists():
        raise FileNotFoundError(f"Directory not found: {path}")

    if not directory.is_dir():
        raise NotADirectoryError(f"Not a directory: {path}")

    files = []

    for item in sorted(directory.iterdir()):
        if item.is_dir():
            files.append(f"[DIR]  {item.name}")
        else:
            files.append(f"[FILE] {item.name}")

    if not files:
        return "Directory is empty."

    return "\n".join(files)

In [57]:
print(list_files())

[DIR]  .ipynb_checkpoints
[FILE] hello.py


In [24]:
def read_file(path: str) -> str:
    file_path = resolve_path(path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    if not file_path.is_file():
        raise IsADirectoryError(f"Not a file: {path}")

    return file_path.read_text(encoding="utf-8")

In [59]:
print(read_file("hello.py"))

print("Hello from the coding agent!")


In [25]:
def write_file(path: str, content: str) -> str:
    file_path = resolve_path(path)

    file_path.parent.mkdir(parents=True, exist_ok=True)

    file_path.write_text(
        content,
        encoding="utf-8"
    )

    return f"Successfully wrote {path}"

In [61]:
print(
    write_file(
        "test.py",
        'print("Created by the agent!")'
    )
)

Successfully wrote test.py


In [62]:
print(list_files())

[DIR]  .ipynb_checkpoints
[FILE] hello.py
[FILE] test.py


In [26]:
def search_files(query: str) -> str:
    results = []

    for file_path in WORKSPACE.rglob("*"):

        if not file_path.is_file():
            continue

        try:
            content = file_path.read_text(encoding="utf-8")
        except (UnicodeDecodeError, PermissionError):
            continue

        for line_number, line in enumerate(
            content.splitlines(),
            start=1
        ):
            if query.lower() in line.lower():
                relative_path = file_path.relative_to(WORKSPACE)

                results.append(
                    f"{relative_path}:{line_number}: {line.strip()}"
                )

    if not results:
        return f"No matches found for: {query}"

    return "\n".join(results)

In [64]:
print(search_files("print"))

hello.py:1: print("Hello from the coding agent!")
test.py:1: print("Created by the agent!")
.ipynb_checkpoints\hello-checkpoint.py:1: print("Hello from the coding agent!")
.ipynb_checkpoints\test-checkpoint.py:1: print("Created by the agent!")


In [27]:
list_files_tool = Tool(
    name="list_files",
    description="Lists files and directories inside the workspace.",
    function=list_files
)

read_file_tool = Tool(
    name="read_file",
    description="Reads and returns the contents of a file inside the workspace.",
    function=read_file
)

write_file_tool = Tool(
    name="write_file",
    description="Creates or overwrites a file inside the workspace with the provided content.",
    function=write_file
)

search_files_tool = Tool(
    name="search_files",
    description="Searches for text inside files in the workspace and returns matching file paths and line numbers.",
    function=search_files
)

In [51]:
tool_registry.register(list_files_tool)
tool_registry.register(read_file_tool)
tool_registry.register(write_file_tool)
tool_registry.register(search_files_tool)

In [29]:
print([
    tool.name
    for tool in tool_registry.list_tools()
])

['add_numbers', 'multiply_numbers', 'list_files', 'read_file', 'write_file', 'search_files']


In [74]:
contents = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(
                text="""
Look at the files in the workspace.
Read hello.py.
Then create a file called summary.txt containing
a short description of what hello.py does.
"""
            )
        ]
    )
]

In [30]:
config = types.GenerateContentConfig(
    tools=[tool_registry.to_gemini_tool()],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    )
)

In [75]:
while True:

    response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=config,
    )

    model_content = response.candidates[0].content
    contents.append(model_content)

    tool_calls = [
        part.function_call
        for part in model_content.parts
        if part.function_call
    ]

    if not tool_calls:

        for part in model_content.parts:
            if part.text:
                print("FINAL:", part.text)

        break

    tool_results = []

    for function_call in tool_calls:

        print("TOOL CALL:")
        print("  name:", function_call.name)
        print("  args:", function_call.args)

        result = tool_registry.execute(
            function_call.name,
            **function_call.args
        )

        print("TOOL RESULT:", result)

        tool_results.append(
            types.Part.from_function_response(
                name=function_call.name,
                response={"result": result}
            )
        )

    contents.append(
        types.Content(
            role="user",
            parts=tool_results
        )
    )

TOOL CALL:
  name: list_files
  args: {}
TOOL RESULT: [DIR]  .ipynb_checkpoints
[FILE] hello.py
[FILE] test.py
TOOL CALL:
  name: read_file
  args: {'path': 'hello.py'}
TOOL RESULT: print("Hello from the coding agent!")
TOOL CALL:
  name: write_file
  args: {'content': 'hello.py is a simple Python script that prints the greeting message "Hello from the coding agent!" to the standard output.', 'path': 'summary.txt'}
TOOL RESULT: Successfully wrote summary.txt
FINAL: I have checked the files in the workspace, read `hello.py`, and created `summary.txt` with a description of what `hello.py` does.


In [31]:
def edit_file(path: str, old_text: str, new_text: str) -> str:
    file_path = resolve_path(path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    if not file_path.is_file():
        raise IsADirectoryError(f"Not a file: {path}")

    content = file_path.read_text(encoding="utf-8")

    if old_text not in content:
        raise ValueError(
            f"Could not find the specified text in {path}"
        )

    occurrences = content.count(old_text)

    if occurrences > 1:
        raise ValueError(
            f"The specified text appears {occurrences} times in {path}. "
            "Provide a more specific piece of text."
        )

    new_content = content.replace(old_text, new_text)

    file_path.write_text(
        new_content,
        encoding="utf-8"
    )

    return f"Successfully edited {path}"

In [32]:
edit_file_tool = Tool(
    name="edit_file",
    description=(
        "Edits a file by replacing one specific piece of text "
        "with new text. The old text must uniquely identify the "
        "section being changed."
    ),
    function=edit_file
)

In [52]:
tool_registry.register(edit_file_tool)

In [34]:
edit_file(
    "hello.py",
    'print("Hello from the coding agent!")',
    'print("Hello from v0!")'
)

ValueError: Could not find the specified text in hello.py

In [85]:
contents = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(
                text="""
Read hello.py.

Change the message from:
"Hello from my coding agent!"

to:
"Checking editting from my coding agent!"

Do not rewrite the entire file. Use the appropriate file editing tool.
"""
            )
        ]
    )
]

In [86]:
while True:

    response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=config,
    )

    model_content = response.candidates[0].content
    contents.append(model_content)

    tool_calls = [
        part.function_call
        for part in model_content.parts
        if part.function_call
    ]

    if not tool_calls:

        for part in model_content.parts:
            if part.text:
                print("FINAL:", part.text)

        break

    tool_results = []

    for function_call in tool_calls:

        print("TOOL CALL:")
        print("  name:", function_call.name)
        print("  args:", function_call.args)

        result = tool_registry.execute(
            function_call.name,
            **function_call.args
        )

        print("TOOL RESULT:", result)

        tool_results.append(
            types.Part.from_function_response(
                name=function_call.name,
                response={"result": result}
            )
        )

    contents.append(
        types.Content(
            role="user",
            parts=tool_results
        )
    )

TOOL CALL:
  name: read_file
  args: {'path': 'hello.py'}
TOOL RESULT: print("Hello from my coding agent!")
TOOL CALL:
  name: edit_file
  args: {'old_text': 'print("Hello from my coding agent!")', 'path': 'hello.py', 'new_text': 'print("Checking editting from my coding agent!")'}
TOOL RESULT: Successfully edited hello.py
FINAL: I have successfully updated the message in `hello.py` from `"Hello from my coding agent!"` to `"Checking editting from my coding agent!"` using the `edit_file` tool without rewriting the entire file.


In [32]:
contents = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(
                text="""
Replace hello.py content.

Write a fizzbuzz program in it that takes n=10
"""
            )
        ]
    )
]

In [34]:
while True:

    response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=config,
    )

    model_content = response.candidates[0].content
    contents.append(model_content)

    tool_calls = [
        part.function_call
        for part in model_content.parts
        if part.function_call
    ]

    if not tool_calls:

        for part in model_content.parts:
            if part.text:
                print("FINAL:", part.text)

        break

    tool_results = []

    for function_call in tool_calls:

        print("TOOL CALL:")
        print("  name:", function_call.name)
        print("  args:", function_call.args)

        result = tool_registry.execute(
            function_call.name,
            **function_call.args
        )

        print("TOOL RESULT:", result)

        tool_results.append(
            types.Part.from_function_response(
                name=function_call.name,
                response={"result": result}
            )
        )

    contents.append(
        types.Content(
            role="user",
            parts=tool_results
        )
    )

TOOL CALL:
  name: write_file
  args: {'path': 'hello.py', 'content': 'for i in range(1, 11):\n    if i % 3 == 0 and i % 5 == 0:\n        print("FizzBuzz")\n    elif i % 3 == 0:\n        print("Fizz")\n    elif i % 5 == 0:\n        print("Buzz")\n    else:\n        print(i)\n'}
TOOL RESULT: Successfully wrote hello.py
FINAL: I have replaced the content of `hello.py` with a FizzBuzz program that loops up to $n = 10$.


In [1]:
import subprocess


def run_command(command: str) -> str:
    result = subprocess.run(
        command,
        shell=True,
        cwd=WORKSPACE,
        capture_output=True,
        text=True,
        timeout=30,
    )

    output = []

    if result.stdout:
        output.append(f"STDOUT:\n{result.stdout}")

    if result.stderr:
        output.append(f"STDERR:\n{result.stderr}")

    output.append(f"EXIT CODE: {result.returncode}")

    return "\n".join(output)

In [5]:
print(run_command("python hello.py"))

STDOUT:
1
2
Fizz
4
Buzz
Fizz
7
8
Fizz
Buzz

EXIT CODE: 0


In [40]:
run_command_tool = Tool(
    name="run_command",
    description=(
        "Executes a shell command inside the workspace. "
        "Use this tool when you need to actually RUN or TEST code. "
        "Reading a file does NOT verify that the code executes correctly. "
        "For Python files, use commands such as `python filename.py`. "
        "Always use this tool when the user explicitly asks you to "
        "run, execute, test, or verify code by execution."
    ),
    function=run_command
)

In [41]:
tool_registry.register(run_command_tool)

In [42]:
print([
    tool.name
    for tool in tool_registry.list_tools()
])

['add_numbers', 'multiply_numbers', 'list_files', 'read_file', 'write_file', 'search_files', 'edit_file', 'run_command']


In [45]:
config = types.GenerateContentConfig(
    tools=[tool_registry.to_gemini_tool()],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    )
)

In [46]:
contents = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(
                text="""
Create a file called calculator.py.

It should contain a function called add(a, b)
that returns a + b.

Then run the file to make sure it has no syntax errors.
"""
            )
        ]
    )
]

In [47]:
while True:

    response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=config,
    )

    model_content = response.candidates[0].content
    contents.append(model_content)

    tool_calls = [
        part.function_call
        for part in model_content.parts
        if part.function_call
    ]

    if not tool_calls:

        for part in model_content.parts:
            if part.text:
                print("FINAL:", part.text)

        break

    tool_results = []

    for function_call in tool_calls:

        print("TOOL CALL:")
        print("  name:", function_call.name)
        print("  args:", function_call.args)

        result = tool_registry.execute(
            function_call.name,
            **function_call.args
        )

        print("TOOL RESULT:", result)

        tool_results.append(
            types.Part.from_function_response(
                name=function_call.name,
                response={"result": result}
            )
        )

    contents.append(
        types.Content(
            role="user",
            parts=tool_results
        )
    )

TOOL CALL:
  name: write_file
  args: {'path': 'calculator.py', 'content': 'def add(a, b):\n    return a + b\n'}
TOOL RESULT: Successfully wrote calculator.py
TOOL CALL:
  name: run_command
  args: {'command': 'python calculator.py'}
TOOL RESULT: EXIT CODE: 0
FINAL: I have created `calculator.py` with the `add(a, b)` function and successfully ran it to verify there are no syntax errors.


In [57]:
while True:

    response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=config,
    )

    model_content = response.candidates[0].content
    contents.append(model_content)

    tool_calls = [
        part.function_call
        for part in model_content.parts
        if part.function_call
    ]

    if not tool_calls:

        for part in model_content.parts:
            if part.text:
                print("FINAL:", part.text)

        break

    tool_results = []

    for function_call in tool_calls:

        print("TOOL CALL:")
        print("  name:", function_call.name)
        print("  args:", function_call.args)

        tool_result = tool_registry.execute(
            function_call.name,
            **function_call.args
        )
        
        print("TOOL RESULT:", tool_result)
        
        tool_results.append(
            types.Part.from_function_response(
                name=function_call.name,
                response={
                    "tool_result": tool_result
                }
            )
        )

    contents.append(
        types.Content(
            role="user",
            parts=tool_results
        )
    )

TOOL CALL:
  name: read_file
  args: {'path': 'meow.txt'}
TOOL RESULT: {'success': False, 'error': 'FileNotFoundError', 'message': 'File not found: meow.txt'}
TOOL CALL:
  name: list_files
  args: {'path': '.'}
TOOL RESULT: {'success': True, 'result': '[DIR]  .ipynb_checkpoints\n[FILE] calculator.py\n[FILE] hello.py\n[FILE] summary.txt\n[FILE] test.py'}
TOOL CALL:
  name: search_files
  args: {'query': 'meow'}
TOOL RESULT: {'success': True, 'result': 'No matches found for: meow'}
FINAL: The file `meow.txt` does not exist in the workspace. Here are the files currently available:

- `calculator.py`
- `hello.py`
- `summary.txt`
- `test.py`

Would you like me to read one of these files instead?


In [56]:
contents = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(
                text="""
Read the file meow.txt in the workspace.
"""
            )
        ]
    )
]

In [58]:
from dataclasses import dataclass
from typing import Callable, Any, get_type_hints
import inspect

from google.genai import types


@dataclass
class Tool:
    name: str
    description: str
    function: Callable[..., Any]

    def execute(self, **kwargs):
        self.validate_arguments(kwargs)
        return self.function(**kwargs)

    def validate_arguments(self, arguments: dict):
        signature = inspect.signature(self.function)
        type_hints = get_type_hints(self.function)

        # Check required arguments
        for name, parameter in signature.parameters.items():

            if (
                parameter.default is inspect.Parameter.empty
                and name not in arguments
            ):
                raise ValueError(
                    f"Missing required argument: {name}"
                )

        # Check argument types
        for name, value in arguments.items():

            if name not in type_hints:
                continue

            expected_type = type_hints[name]

            if not isinstance(value, expected_type):
                raise TypeError(
                    f"Argument '{name}' must be "
                    f"{expected_type.__name__}, "
                    f"got {type(value).__name__}"
                )

    def get_parameters(self):
        signature = inspect.signature(self.function)
        type_hints = get_type_hints(self.function)

        properties = {}
        required = []

        for param_name, param in signature.parameters.items():

            if param.kind in (
                inspect.Parameter.VAR_POSITIONAL,
                inspect.Parameter.VAR_KEYWORD,
            ):
                continue

            python_type = type_hints.get(param_name, str)

            if python_type == int:
                json_type = "integer"
            elif python_type == float:
                json_type = "number"
            elif python_type == bool:
                json_type = "boolean"
            else:
                json_type = "string"

            properties[param_name] = {
                "type": json_type
            }

            if param.default is inspect.Parameter.empty:
                required.append(param_name)

        return {
            "type": "object",
            "properties": properties,
            "required": required,
        }

    def to_gemini_declaration(self):
        parameters = self.get_parameters()

        return types.FunctionDeclaration(
            name=self.name,
            description=self.description,
            parameters=types.Schema(
                type="OBJECT",
                properties={
                    name: types.Schema(
                        type=info["type"].upper()
                    )
                    for name, info in parameters["properties"].items()
                },
                required=parameters["required"],
            ),
        )

In [61]:
class ToolRegistry:

    def __init__(self):
        self.tools = {}

    def register(self, tool: Tool):
        self.tools[tool.name] = tool

    def get(self, name: str) -> Tool:
        if name not in self.tools:
            raise KeyError(f"Unknown tool: {name}")

        return self.tools[name]

    def list_tools(self):
        return list(self.tools.values())

    def execute(self, name: str, **kwargs):
        try:
            tool = self.get(name)

            result = tool.execute(**kwargs)

            return {
                "success": True,
                "result": result
            }

        except Exception as e:

            return {
                "success": False,
                "error": type(e).__name__,
                "message": str(e)
            }

    def to_gemini_tool(self):
        return types.Tool(
            function_declarations=[
                tool.to_gemini_declaration()
                for tool in self.tools.values()
            ]
        )

## Tool Argument Validation

An LLM generates tool arguments, but the LLM's output cannot be
trusted to automatically satisfy the application's requirements.

The agent therefore has two separate responsibilities:

1. The LLM decides which tool to call and proposes arguments.
2. The runtime validates those arguments before executing Python code.

The flow is:

LLM
->
Function Call
->
Argument Validation
->
Tool Execution
->
Tool Result
->
LLM

This creates a trust boundary between model-generated data and
application code.

#### Context Management:

In [63]:
contents = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(
                text="Create test.py that prints hello"
            )
        ],
    )
]

print(contents)

[Content(
  parts=[
    Part(
      text='Create test.py that prints hello'
    ),
  ],
  role='user'
)]


In [64]:
response = client.models.generate_content(
    model=MODEL,
    contents=contents,
    config=config,
)

In [65]:
model_content = response.candidates[0].content

contents.append(model_content)

print("Number of messages:", len(contents))

for i, content in enumerate(contents):
    print(f"\n--- Message {i} ---")
    print("Role:", content.role)

    for part in content.parts:
        if part.text:
            print("TEXT:", part.text)

        if part.function_call:
            print(
                "FUNCTION CALL:",
                part.function_call.name,
                part.function_call.args,
            )

Number of messages: 2

--- Message 0 ---
Role: user
TEXT: Create test.py that prints hello

--- Message 1 ---
Role: model
FUNCTION CALL: write_file {'content': 'print("hello")\n', 'path': 'test.py'}
